# Day 2 Session 3 -- Exercises: LangGraph Orchestration & Workflow Design

Work through the exercises below to practice building LangGraph workflows. You have the demos notebook as reference.

**Engineering context:** You are an engineer building workflow orchestration systems. Your users (consultants) need reliable, stateful pipelines that route work intelligently, refine outputs iteratively, and include human checkpoints. LangGraph lets you model these requirements as directed graphs with typed state, conditional edges, and cycles.

**Structure:**
- Exercises 1-3: Core (complete these during the session)
- Optional Exercises 1-3: Stretch goals (if you finish early or for post-session practice)

In [ ]:
!pip install -q langchain langchain-openai langchain-core langgraph python-dotenv

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, Literal
import operator
import json
import os

print("All imports successful!")

## Recap

In the demos you saw five LangGraph patterns:
1. **Linear pipelines** with typed state and sequential edges
2. **Conditional routing** that branches based on LLM classification
3. **ReAct think-act-observe loop** for tool-using agents
4. **Cyclic refinement** that iterates until quality thresholds are met
5. **Human-in-the-loop gates** for approval checkpoints

These exercises will have you build each pattern yourself, starting simple and building toward a combined system.

---
## Exercise 1 (Easy): Client Onboarding Pipeline (3-Node Linear StateGraph)

**Reference:** Demo 1 in the demos notebook

> **Hint:** Each chain step feeds its output as input to the next. Build and test each step independently first.

**Scenario:** When a new client engagement kicks off, the team runs a standard onboarding pipeline:
1. **gather_data** -- Clean and normalize the raw client data (remove extra whitespace)
2. **analyze_situation** -- Use an LLM to generate a preliminary diagnostic summary highlighting key challenges and opportunities (2-3 sentences)
3. **prepare_brief** -- Use an LLM to transform the diagnostic into an executive-ready brief suitable for partner review (2-3 sentences)

**Your task:** Build a three-node linear StateGraph that processes raw client data through these three steps.

### Step-by-Step Guide

1. Define an `OnboardingState` TypedDict with fields: `raw_client_data`, `clean_data`, `diagnostic_summary`, `executive_brief`
2. Initialize the LLM with `ChatOpenAI`
3. Write `gather_data` node -- pure Python, no LLM needed. Clean whitespace with `" ".join(state["raw_client_data"].split())`
4. Write `analyze_situation` node -- call `llm.invoke()` with a SystemMessage ("You are a McKinsey consultant. Analyze this client data and provide a 2-3 sentence diagnostic summary highlighting key challenges and opportunities.") and a HumanMessage (the clean_data)
5. Write `prepare_brief` node -- call `llm.invoke()` with a SystemMessage ("You are a McKinsey engagement manager. Transform this diagnostic into a 2-3 sentence executive brief suitable for partner review.") and a HumanMessage (the diagnostic_summary)
6. Build the StateGraph: add nodes, set entry point, add edges (gather_data -> analyze_situation -> prepare_brief -> END)
7. Compile and invoke with the test data

In [ ]:
# Exercise 1 - Client Onboarding Pipeline

# Step 1: State definition (provided)
class OnboardingState(TypedDict):
    raw_client_data: str
    clean_data: str
    diagnostic_summary: str
    executive_brief: str

# Step 2: Initialize LLM (provided)
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)

# Step 3: gather_data node (provided -- no LLM needed, just string cleanup)
def gather_data(state: OnboardingState) -> dict:
    """Clean and normalize the raw client data."""
    return {"clean_data": " ".join(state["raw_client_data"].split())}

# TODO Step 4: Define analyze_situation node
# Hint: Use llm.invoke([SystemMessage("You are a McKinsey consultant. Summarize the key
#       challenges and opportunities in 2-3 sentences."), HumanMessage(content=state["clean_data"])])
# Return {"diagnostic_summary": response.content}

def analyze_situation(state: OnboardingState) -> dict:
    # YOUR CODE HERE
    pass

# TODO Step 5: Define prepare_brief node
# Hint: Same pattern as above but with a different system message asking for
#       a 3-bullet executive brief. Input is state["diagnostic_summary"].
# Return {"executive_brief": response.content}

def prepare_brief(state: OnboardingState) -> dict:
    # YOUR CODE HERE
    pass

# Step 6: Build the StateGraph (provided -- uncomment after writing nodes)
# graph = StateGraph(OnboardingState)
# graph.add_node("gather_data", gather_data)
# graph.add_node("analyze_situation", analyze_situation)
# graph.add_node("prepare_brief", prepare_brief)
# graph.set_entry_point("gather_data")
# graph.add_edge("gather_data", "analyze_situation")
# graph.add_edge("analyze_situation", "prepare_brief")
# graph.add_edge("prepare_brief", END)
# app = graph.compile()

# Test (uncomment after graph is built)
# result = app.invoke({
#     "raw_client_data": "   Client:  Meridian Health    Revenue: $4.2B   Challenge:  margin  compression   due  to  rising   labor  costs   and   payer   mix   shift   toward   Medicaid.   Exploring   digital  health   to   offset.  ",
#     "clean_data": "", "diagnostic_summary": "", "executive_brief": ""
# })
# print("=== Diagnostic Summary ===")
# print(result["diagnostic_summary"])
# print("\n=== Executive Brief ===")
# print(result["executive_brief"])

### Progressive Hints

<details>
<summary>Hint 1: State definition</summary>

```python
class OnboardingState(TypedDict):
    raw_client_data: str
    clean_data: str
    diagnostic_summary: str
    executive_brief: str
```
</details>

<details>
<summary>Hint 2: Node function pattern</summary>

```python
def analyze_situation(state: OnboardingState) -> dict:
    response = llm.invoke([
        SystemMessage(content="Your system prompt here"),
        HumanMessage(content=state["clean_data"])
    ])
    return {"diagnostic_summary": response.content}
```
</details>

<details>
<summary>Hint 3: Graph construction pattern</summary>

Remember from Demo 1:
- `graph.set_entry_point("first_node")` -- where execution starts
- `graph.add_edge("node_a", "node_b")` -- sequential connection
- `graph.add_edge("last_node", END)` -- terminates the graph
</details>

### Expected Output

When you run the cell above with the provided test data, you should see output similar to:

```
Clean Data: Client: GlobalRetail Corp. Revenue $8.2B, declining margins from 12% to 7% over 3 years. Major cost pressures in supply chain and labor. Considering digital transformation but lacks internal capabilities. Competitors gaining share in e-commerce channel.

Diagnostic: GlobalRetail Corp. faces a margin compression challenge driven by rising supply chain
and labor costs, compounded by competitive losses in the e-commerce channel. The company's interest
in digital transformation is well-placed but will require external capability building given the
current internal gap. Priority areas for investigation include supply chain cost reduction
opportunities, digital channel strategy, and build-vs-buy decisions for technology capabilities.

Executive Brief: GlobalRetail Corp., an $8.2B retailer experiencing significant margin erosion (12%
to 7% over three years), requires an integrated diagnostic spanning supply chain optimization,
digital transformation readiness, and competitive response strategy. We recommend a 10-week
engagement to quantify cost reduction levers and develop a phased digital transformation roadmap
with clear capability-building milestones.
```

The exact wording will vary (LLM output is non-deterministic), but you should see:
- **Clean Data** with normalized whitespace (no extra spaces)
- **Diagnostic** summarizing challenges and opportunities in 2-3 sentences
- **Executive Brief** in polished consulting language suitable for partner review

---
## Exercise 2 (Easy): Conditional Routing Pipeline

**Reference:** Demo 2 in the demos notebook

> **Hint:** Each chain step feeds its output as input to the next. Build and test each step independently first.

**Scenario:** You are building a consulting question router. Incoming questions from associates need to be classified and routed to the right specialist for a focused response. The three specialist areas are:

1. **market_analysis** -- questions about market sizing, competitive landscape, customer segmentation
2. **financial_modeling** -- questions about valuation, financial projections, cost structure
3. **operational_improvement** -- questions about process optimization, supply chain, lean operations

**Your task:** Build a StateGraph that:
1. Takes a consulting question as input
2. Uses an LLM to classify it into one of the three categories
3. Routes to the appropriate specialist node that provides a focused response
4. Outputs the specialist's answer

### Step-by-Step Guide

1. Define a `QuestionRouterState` TypedDict with fields: `question`, `category`, `specialist_response`
2. Write a `classify_question` node that uses the LLM to classify the question (respond with one word: market_analysis, financial_modeling, or operational_improvement)
3. Write three specialist nodes (`market_analyst`, `financial_modeler`, `operations_expert`) -- each uses a different system prompt and returns a focused response
4. Write a `route_to_specialist` function that reads `state["category"]` and returns the appropriate node name
5. Build the graph: classify_question -> (conditional edge) -> specialist_node -> END
6. Test with three different questions (one for each category)

In [ ]:
# Exercise 2 - Conditional Routing Pipeline

# Step 1: State (provided)
class QuestionRouterState(TypedDict):
    question: str
    category: str
    specialist_response: str

# Step 2: LLM (provided)
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)

# Step 3: Classify node (provided)
def classify_question(state: QuestionRouterState) -> dict:
    """Classify the question into one of three categories."""
    response = llm.invoke([
        SystemMessage(content="Classify this consulting question into exactly one category: "
                      "'market_analysis', 'financial_modeling', or 'operational_improvement'. "
                      "Respond with just the category name."),
        HumanMessage(content=state["question"])
    ])
    return {"category": response.content.strip().lower()}

# TODO Step 4: Define 3 specialist nodes
# Each takes state and returns {"specialist_response": response.content}
# Hint: Use a SystemMessage describing the specialist role + HumanMessage with the question

def market_specialist(state: QuestionRouterState) -> dict:
    # YOUR CODE HERE -- SystemMessage: "You are a market analysis expert..."
    pass

def financial_specialist(state: QuestionRouterState) -> dict:
    # YOUR CODE HERE -- SystemMessage: "You are a financial modeling expert..."
    pass

def operations_specialist(state: QuestionRouterState) -> dict:
    # YOUR CODE HERE -- SystemMessage: "You are an operations improvement expert..."
    pass

# Step 5: Routing function (provided)
def route_question(state: QuestionRouterState) -> str:
    cat = state["category"]
    if "market" in cat: return "market"
    if "financial" in cat: return "financial"
    return "operations"

# Step 6: Build graph (provided -- uncomment after writing specialist nodes)
# graph = StateGraph(QuestionRouterState)
# graph.add_node("classify", classify_question)
# graph.add_node("market", market_specialist)
# graph.add_node("financial", financial_specialist)
# graph.add_node("operations", operations_specialist)
# graph.set_entry_point("classify")
# graph.add_conditional_edges("classify", route_question, {
#     "market": "market", "financial": "financial", "operations": "operations"
# })
# graph.add_edge("market", END)
# graph.add_edge("financial", END)
# graph.add_edge("operations", END)
# app = graph.compile()

# Test
# for q in [
#     "What is the total addressable market for EV charging in Europe?",
#     "Build a DCF model for a SaaS company with 30% revenue growth",
#     "How can we reduce warehouse picking time by 25%?"
# ]:
#     result = app.invoke({"question": q, "category": "", "specialist_response": ""})
#     print(f"Q: {q[:60]}...")
#     print(f"Category: {result['category']}")
#     print(f"Response: {result['specialist_response'][:150]}...")
#     print()

### Progressive Hints

<details>
<summary>Hint 1: Routing function pattern</summary>

```python
def route_to_specialist(state: QuestionRouterState) -> str:
    category = state["category"]
    if "market" in category:
        return "market_analyst"
    elif "financial" in category:
        return "financial_modeler"
    else:
        return "operations_expert"
```
</details>

<details>
<summary>Hint 2: Conditional edges setup</summary>

```python
graph.add_conditional_edges("classify_question", route_to_specialist, {
    "market_analyst": "market_analyst",
    "financial_modeler": "financial_modeler",
    "operations_expert": "operations_expert"
})
```
</details>

<details>
<summary>Hint 3: Complete specialist node example</summary>

```python
def market_analyst(state: QuestionRouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a market analysis specialist at McKinsey. Provide a structured response with market size estimate, key players, and growth trends. Keep it to 3-4 sentences."),
        HumanMessage(content=state["question"])
    ])
    return {"specialist_response": f"[MARKET ANALYSIS]\n{response.content}"}
```
</details>

### Expected Output

You should see each question classified correctly and routed to the appropriate specialist:

```
Question: What is the total addressable market for electric vehicles in Southeast Asia?
Category: market_analysis
Response: [MARKET ANALYSIS]
The Southeast Asia EV market is estimated at $X billion...
============================================================
Question: How should we value a SaaS startup with $5M ARR growing at 40% YoY?
Category: financial_modeling
Response: [FINANCIAL MODELING]
For a SaaS startup with $5M ARR and 40% growth...
============================================================
Question: How can we reduce manufacturing cycle time by 30% without increasing headcount?
Category: operational_improvement
Response: [OPERATIONAL IMPROVEMENT]
To achieve a 30% cycle time reduction...
============================================================
```

---
## Exercise 3 (Easy): Iterative Report Writer

**Reference:** Demo 4 in the demos notebook

> **Hint:** The RAG pattern is: embed query -> search vector store -> format context -> send to LLM with context + question.

**Scenario:** You are building an iterative report generation system. The workflow is:
1. **Draft** a consulting report section based on a topic
2. **Quality review** -- an LLM reviewer scores the draft 1-10 on clarity, depth, and actionability
3. **Decision** -- if average score >= 8, output the final version. If < 8 OR max 3 iterations reached, refine and loop back.

This models the real consulting workflow where deliverables go through multiple drafts before reaching client-ready quality.

**Your task:** Build a StateGraph with a cycle that refines a report draft until quality is sufficient.

### Step-by-Step Guide

1. Define a `ReportState` TypedDict with fields: `topic`, `draft`, `review_feedback`, `score`, `iteration`, `is_final`
2. Write a `draft_report` node -- if no existing draft, create one from scratch; if feedback exists, refine the draft based on feedback
3. Write a `review_report` node -- LLM scores the draft 1-10 on clarity, depth, and actionability. Parse the average score. Set `is_final = True` if score >= 8 or iteration >= 3
4. Write a `should_continue` routing function -- returns "draft_report" if not final, "end" if final
5. Build the graph: draft_report -> review_report -> (conditional: draft_report or END)
6. Test with a consulting topic

In [ ]:
# Exercise 3 - Iterative Report Writer

# Step 1: State (provided)
class ReportState(TypedDict):
    topic: str
    draft: str
    review_feedback: str
    score: float
    iteration: int
    is_final: bool

# Step 2: LLM (provided)
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0.7)

# Step 3: Draft node (provided)
def draft_report(state: ReportState) -> dict:
    """Generate or refine a report draft."""
    if not state.get("draft"):
        # First draft
        response = llm.invoke([
            SystemMessage(content="You are a McKinsey consultant. Write a concise consulting report section (3-4 paragraphs) on the given topic. Be specific, data-driven, and actionable."),
            HumanMessage(content=f"Write a report on: {state['topic']}")
        ])
    else:
        # Refine based on feedback
        response = llm.invoke([
            SystemMessage(content="You are a McKinsey consultant. Refine this report based on the reviewer's feedback. Keep the same structure but improve quality."),
            HumanMessage(content=f"Current draft:\n{state['draft']}\n\nFeedback:\n{state['review_feedback']}\n\nWrite an improved version.")
        ])
    return {"draft": response.content, "iteration": state.get("iteration", 0) + 1}

# TODO Step 4: Define review_report node
# Use a reviewer LLM (temperature=0) to score the draft 1-10 on clarity, depth, actionability.
# Hint: Ask the reviewer to respond in format "Score: X/10\nFeedback: ..."
#       Parse the score with a regex or simple string split.
# Return {"review_feedback": feedback, "score": avg_score}

def review_report(state: ReportState) -> dict:
    reviewer = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)
    # YOUR CODE HERE
    pass

# TODO Step 5: Define should_continue routing function
# Return "draft" if score < 8 and iteration < 3, else "end"

def should_continue(state: ReportState) -> str:
    # YOUR CODE HERE
    pass

# Step 6: Build graph (provided -- uncomment after writing review + routing)
# graph = StateGraph(ReportState)
# graph.add_node("draft", draft_report)
# graph.add_node("review", review_report)
# graph.set_entry_point("draft")
# graph.add_edge("draft", "review")
# graph.add_conditional_edges("review", should_continue, {"draft": "draft", "end": END})
# app = graph.compile()

# Test
# result = app.invoke({
#     "topic": "Digital transformation strategy for a mid-size regional bank",
#     "draft": "", "review_feedback": "", "score": 0.0, "iteration": 0, "is_final": False
# })
# print(f"Final draft (iteration {result['iteration']}, score {result['score']}):")
# print(result["draft"][:500])

### Progressive Hints

<details>
<summary>Hint 1: Score parsing approach</summary>

```python
import re

# Look for 'Average: X/10' or 'Average: X' pattern in the response
match = re.search(r'Average:\s*(\d+\.?\d*)', response.content)
if match:
    score = float(match.group(1))
else:
    score = 5.0  # Default if parsing fails
```
</details>

<details>
<summary>Hint 2: Draft refinement prompt structure</summary>

```python
if state["draft"]:
    prompt = f"""Refine this report based on reviewer feedback.

Current draft:
{state['draft']}

Reviewer feedback:
{state['review_feedback']}

Provide the improved version (3-4 paragraphs):"""
else:
    prompt = f"Write a consulting report section on: {state['topic']}"
```
</details>

<details>
<summary>Hint 3: Complete review node pattern</summary>

```python
def review_report(state: ReportState) -> dict:
    response = llm.invoke([
        SystemMessage(content="Score this report 1-10 on Clarity, Depth, Actionability. Format: 'Average: X/10'. Then give improvement suggestions."),
        HumanMessage(content=state["draft"])
    ])
    match = re.search(r'Average:\s*(\d+\.?\d*)', response.content)
    score = float(match.group(1)) if match else 5.0
    is_final = score >= 8 or state["iteration"] >= 3
    print(f"  [Review] Score: {score}/10 | Final: {is_final}")
    return {"review_feedback": response.content, "score": score, "is_final": is_final}
```
</details>

### Expected Output

You should see the report improve over iterations:

```
  [Iteration 1] Drafting report...
  [Review] Score: 6.0/10 | Final: False
  [Iteration 2] Refining based on feedback...
  [Review] Score: 7.5/10 | Final: False
  [Iteration 3] Refining based on feedback...
  [Review] Score: 8.0/10 | Final: True

Final Report (after 3 iterations, score: 8.0):
[A well-structured report with specific data points, clear recommendations, and timelines...]
```

---
---
## Optional Exercises (Intermediate)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
These are stretch goals for students who finish the core exercises early, or for post-session practice.

---
## Optional Exercise 1 (Intermediate): ReAct Research Agent

**Reference:** Demo 3 in the demos notebook


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
**Scenario:** Build a ReAct agent that researches consulting topics using three simulated tools:
- `market_data(query)` -- returns market sizing and growth data
- `competitor_info(query)` -- returns competitive intelligence
- `industry_trends(query)` -- returns industry trend analysis

The agent should think about what information it needs, choose the right tool, observe the result, and repeat until it can synthesize a complete answer (or hits 5 steps).

**Your task:** Implement the full ReAct think-act-observe cycle as a LangGraph with:
- A state that tracks thoughts, actions, observations, and step count
- A think node that reasons about the next research action
- An act node that dispatches to the correct tool
- A conditional edge that decides whether to continue or stop

In [ ]:
# Optional Exercise 1 - ReAct Research Agent

# Simulated tools (provided for you)
def market_data(query: str) -> str:
    """Simulated market data tool."""
    data = {
        "cloud computing": "Global cloud market: $600B (2024), growing 20% CAGR. AWS 32%, Azure 23%, GCP 11% market share.",
        "ai software": "AI software market: $150B (2024), projected $400B by 2028. Enterprise AI adoption at 55%.",
        "cybersecurity": "Cybersecurity market: $180B (2024), growing 12% CAGR. Driven by cloud migration and regulatory requirements."
    }
    for key, val in data.items():
        if key in query.lower():
            return val
    return f"No market data for: {query}"

def competitor_info(query: str) -> str:
    """Simulated competitor intelligence tool."""
    data = {
        "cloud providers": "Top cloud providers: AWS (most mature), Azure (enterprise integration), GCP (AI/ML focus), Oracle (database workloads).",
        "ai companies": "Key AI players: OpenAI (GPT), Google (Gemini), Anthropic (Claude), Microsoft (Copilot), Meta (Llama open-source).",
        "consulting firms": "Top strategy firms: McKinsey, BCG, Bain (MBB). Growing competition from Big4 strategy arms and boutiques."
    }
    for key, val in data.items():
        if key in query.lower():
            return val
    return f"No competitor info for: {query}"

def industry_trends(query: str) -> str:
    """Simulated industry trends tool."""
    data = {
        "digital transformation": "Key trends: AI-first strategies, platform business models, API ecosystems, composable architecture, and sustainable IT.",
        "financial services": "FinServ trends: embedded finance, real-time payments, AI fraud detection, open banking APIs, and DeFi integration.",
        "healthcare": "Healthcare trends: AI diagnostics, precision medicine, telehealth 2.0, healthcare-at-home, and interoperability standards."
    }
    for key, val in data.items():
        if key in query.lower():
            return val
    return f"No trend data for: {query}"


# TODO: Define ReActResearchState TypedDict
# Fields: question, thoughts (list[str]), actions (list[str]), observations (list[str]),
#         final_answer (str), step_count (int)


# TODO: Define think_node
# The LLM should decide: ACTION: market_data('<query>'), ACTION: competitor_info('<query>'),
# ACTION: industry_trends('<query>'), or FINAL ANSWER: <synthesis>


# TODO: Define act_node
# Parse the action from the latest thought
# Dispatch to the correct tool based on the action name
# Record the observation


# TODO: Define should_continue routing function
# Stop if final_answer is set or step_count >= 5


# TODO: Build graph (think -> act -> conditional -> think or END)


# TODO: Test with: "What is the state of AI in enterprise software and who are the key players?"


---
## Optional Exercise 2 (Intermediate): Pipeline with Human Gate

**Reference:** Demo 5 in the demos notebook


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
**Scenario:** Build a deliverable pipeline with a human approval gate:
1. **draft_deliverable** -- LLM generates a client recommendation
2. **human_review_gate** -- simulates partner review (auto-approves if the draft contains certain quality markers like specific numbers, timelines, and action verbs)
3. **deliver** -- packages the approved deliverable for the client

The gate should check for quality markers rather than always auto-approving. If the draft lacks specific numbers or timelines, it should be rejected.

**Your task:** Build this pipeline with a smart gate that evaluates quality heuristically.

In [ ]:
# Optional Exercise 2 - Pipeline with Human Gate

# TODO: Define DeliverableState TypedDict
# Fields: context (str), draft (str), approved (bool), rejection_reason (str), final_output (str)


# TODO: Define draft_deliverable node
# Use LLM to generate a client recommendation with specific numbers and timelines


# TODO: Define human_review_gate node
# Check for quality markers in the draft:
#   - Contains at least one number (use regex: r'\d+')
#   - Contains a timeline word ("week", "month", "quarter", "year")
#   - Contains an action verb ("implement", "launch", "execute", "deploy", "establish")
# If all three markers present: approve
# Otherwise: reject with specific reason


# TODO: Define deliver node
# If approved: wrap in executive summary format
# If not approved: output the rejection reason


# TODO: Define routing function after review
# If approved -> "deliver", else -> END (blocked)


# TODO: Build graph and test
# Test with: "Recommend a cloud migration strategy for a $2B financial services firm"


---
## Optional Exercise 3 (Intermediate): Multi-Stage Engagement Pipeline

**Reference:** Combines Demo 1 (linear pipeline) + Demo 2 (conditional routing) + Demo 4 (iterative refinement)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
**Scenario:** Build a complete engagement workflow that combines three patterns:

1. **Intake** (linear) -- Clean and normalize the client request
2. **Classify** (conditional routing) -- Determine engagement type and route to the appropriate workflow
3. **Execute** (iterative refinement) -- Generate and refine the deliverable until quality threshold
4. **Output** -- Format the final result

The engagement types are:
- "strategy" -- route to strategy development workflow
- "operations" -- route to operational improvement workflow
- "due_diligence" -- route to M&A due diligence workflow

Each workflow has its own specialized prompts, and all go through iterative refinement (max 2 iterations for this exercise to keep API costs low).

**Your task:** Build the full multi-stage pipeline. This is the most complex graph you will build today.

In [ ]:
# Optional Exercise 3 - Multi-Stage Engagement Pipeline
# This combines linear pipeline + conditional routing + iterative refinement

# TODO: Define EngagementPipelineState TypedDict
# Fields:
#   raw_request: str          -- original client input
#   clean_request: str        -- normalized input
#   engagement_type: str      -- "strategy", "operations", or "due_diligence"
#   draft_output: str         -- current draft
#   review_feedback: str      -- quality feedback
#   iteration: int            -- current iteration count
#   is_complete: bool         -- whether quality threshold met
#   final_deliverable: str    -- formatted final output


# TODO: Define intake_node (linear -- clean the request)


# TODO: Define classify_node (use LLM to classify into strategy/operations/due_diligence)


# TODO: Define route_by_type function (returns node name based on engagement_type)


# TODO: Define strategy_node (drafts strategy recommendation)


# TODO: Define operations_node (drafts operational improvement plan)


# TODO: Define due_diligence_node (drafts M&A assessment)


# TODO: Define quality_review_node (scores 1-10, sets is_complete if >= 8 or iteration >= 2)


# TODO: Define should_refine function (routes back to appropriate workflow or to output)
# CHALLENGE: The routing needs to remember which workflow node to return to!
# Hint: You can use state["engagement_type"] to determine which node to loop back to


# TODO: Define format_output_node (wraps in professional deliverable format)


# TODO: Build the graph
# intake -> classify -> (conditional: strategy/operations/due_diligence) -> quality_review -> (conditional: loop back or format_output) -> END


# TODO: Test with:
# "Our PE portfolio company, a $500M industrial manufacturer, needs to cut costs by 20% in 12 months to meet debt covenants"


### Hints for Optional Exercise 3

<details>
<summary>Hint 1: Handling the refinement loop with routing</summary>

The tricky part is routing BACK to the correct specialist node after quality review. Use state["engagement_type"] in your `should_refine` function:

```python
def should_refine(state) -> str:
    if state["is_complete"]:
        return "format_output"
    # Route back to the appropriate specialist
    type_to_node = {
        "strategy": "strategy_node",
        "operations": "operations_node",
        "due_diligence": "due_diligence_node"
    }
    return type_to_node.get(state["engagement_type"], "format_output")
```
</details>

<details>
<summary>Hint 2: Graph structure</summary>

```
intake -> classify -> [conditional] -> strategy_node  \
                                    -> operations_node  --> quality_review -> [conditional] -> format_output -> END
                                    -> due_diligence_node /                                -> loop back to specialist
```

You need conditional edges in TWO places:
1. After classify (route to specialist)
2. After quality_review (loop back or finish)
</details>

<details>
<summary>Hint 3: Specialist nodes with refinement awareness</summary>

Make each specialist node check if there is existing feedback to refine:

```python
def strategy_node(state) -> dict:
    if state["draft_output"] and state["review_feedback"]:
        prompt = f"Refine this strategy based on feedback:\n{state['draft_output']}\nFeedback: {state['review_feedback']}"
    else:
        prompt = f"Draft a strategy recommendation for: {state['clean_request']}"
    response = llm.invoke([SystemMessage(content="..."), HumanMessage(content=prompt)])
    return {"draft_output": response.content, "iteration": state["iteration"] + 1}
```
</details>

---
## Summary

| Exercise | Pattern | Key Skills Practiced |
|----------|---------|---------------------|
| 1 | Linear Pipeline | TypedDict state, node functions, sequential edges |
| 2 | Conditional Routing | LLM classification, routing functions, `add_conditional_edges` |
| 3 | Iterative Refinement | Cycles, score parsing, termination conditions |
| Optional 1 | ReAct Agent | Think-act-observe loop, tool dispatch, step limits |
| Optional 2 | Human Gate | Quality heuristics, approval/rejection logic |
| Optional 3 | Combined Pipeline | Multi-pattern integration, complex routing, state management |

**Next session:** Multi-Agent Workflows -- building systems where multiple specialized agents collaborate on complex tasks.